## Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual [here](https://quantra.quantinsti.com/quantra-notebook).
2. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
3. Before running this notebook on your local PC:
   - You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on **"Run Codes Locally on Your Machine"** in the course.
   - You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or Python modules that might be required to run this notebook.


# Building a Low-Frequency Trading System

Welcome! So far, we have built a real-time trading system using WebSockets, which is perfect for strategies that need to react to every price tick. However, many successful strategies, like those used by swing or positional traders, are based on slower-moving indicators calculated on daily or hourly data.

In this notebook, we will build a trading system specifically designed for these **low-frequency** strategies. This approach is more efficient and robust for strategies that don't require quick reactions.

## Outline

1.  **The Polling Architecture**: Understanding the difference between WebSockets (push) and Polling (pull).
2.  **Setup & Initialization**: We will only need the `KiteConnect` object for this system.
3.  **The Data & Analysis Function**: Creating a function to fetch historical data and calculate our indicator.
4.  **The Main Trading Loop**: Building the core of our system that runs our logic at a set interval.
5.  **Running the System**: Starting the main loop.
6.  **Conclusion**: Choosing the right architecture for your strategy.

### 1. The Polling Architecture

Instead of listening for data to be *pushed* to us via a WebSocket, we will build a system that periodically *pulls* the data it needs. This is called **polling**.

The architecture is simple:
1.  Wake up.
2.  Fetch the latest historical data.
3.  Perform calculations and check for signals.
4.  Go back to sleep for a defined interval (e.g., one hour).
5.  Repeat.

This is far more efficient for long-term indicators as it saves bandwidth and computing resources. We will not use `KiteTicker` at all.

### 2. Setup & Initialization

First, we'll import the necessary libraries and set up our `KiteConnect` object. This object is our gateway to fetching historical data and, if we choose, placing orders. We also define our API credentials here.

In [1]:
# --- Imports ---
import time
from datetime import datetime, timedelta
from kiteconnect import KiteConnect
import logging

# --- Standard Setup ---
logging.basicConfig(level=logging.INFO)
# IMPORTANT: Replace these with your actual api_key and access_token.
# You get the access_token after a successful login via the Kite Connect API.
api_key = "YOUR_API_KEY"
access_token = "YOUR_ACCESS_TOKEN"

kite = KiteConnect(api_key=api_key)
kite.set_access_token(access_token)

### 3. The Data & Analysis Function

Next, we create the `fetch_data_and_calculate_sma` function. Its job is to connect to the Kite API, download the latest historical data using `kite.historical_data`, and calculate the 200-day Simple Moving Average. It returns the last closing price and the calculated SMA.

In [2]:
# --- Strategy Parameters ---
trading_symbol = 'NSE:RELIANCE'
instrument_token = 738561 # Reliance
interval = 'day'         # We want daily data
sma_period = 200

def fetch_data_and_calculate_sma():
    """Fetches historical data and calculates the SMA."""
    try:
        # For a 200-day MA, we need more than 200 data points.
        to_date = datetime.now().date()
        from_date = to_date - timedelta(days=300) # Fetch ~300 days to be safe
        
        records = kite.historical_data(
            instrument_token=instrument_token,
            from_date=from_date,
            to_date=to_date,
            interval=interval
        )
        
        if len(records) < sma_period:
            logging.warning(f"Not enough data to calculate {sma_period}-day SMA. Found {len(records)} records.")
            return None, None

        # Get the closing prices for the exact period
        close_prices = [r['close'] for r in records[-sma_period:]]
        sma = sum(close_prices) / sma_period
        last_close = records[-1]['close']
        
        return last_close, sma
        
    except Exception as e:
        logging.error(f"Error fetching data or calculating SMA: {e}")
        return None, None

### 4. The Main Trading Loop

This is the heart of our system. The `main_trading_loop` function is an infinite `while True` loop. Inside, it calls our data function, provides a placeholder for your trading logic, and then uses `time.sleep()` to pause execution. The duration of this pause dictates the frequency of our trading system.

In [3]:
# How often to run the check, in seconds.
check_interval_seconds = 3600 # Check once every hour (3600 seconds)

def main_trading_loop():
    """The main polling loop for our strategy."""
    logging.info("Starting low-frequency trading system...")
    while True:
        try:
            logging.info("Running strategy check...")
            last_close, sma = fetch_data_and_calculate_sma()
            
            if last_close is not None and sma is not None:
                logging.info(f"Data Check | Last Close: {last_close} | {sma_period}-Day SMA: {sma:.2f}")
                
                # --- YOUR TRADING LOGIC GOES HERE ---
                # This is where you would check for signals and place orders.
                # Example (no order is placed):
                if last_close > sma:
                    logging.warning("POTENTIAL BUY SIGNAL: Last close is above the SMA.")
                    # place_market_order(...) # You would call your order function here
                elif last_close < sma:
                    logging.warning("POTENTIAL SELL SIGNAL: Last close is below the SMA.")
                    # place_market_order(...) # You would call your order function here
            
            logging.info(f"Check complete. Sleeping for {check_interval_seconds} seconds...")
            time.sleep(check_interval_seconds)
            
        except KeyboardInterrupt:
            # This allows you to stop the system gracefully by pressing Ctrl+C
            logging.info("Trading system stopped by user.")
            break
        except Exception as e:
            # This makes the system more robust to unexpected errors
            logging.error(f"An error occurred in the main loop: {e}")
            logging.info("Waiting for 60 seconds before retrying...")
            time.sleep(60)

### 5. Running the System

To start the system, we simply call our `main_trading_loop()` function. The `if __name__ == "__main__":` block is standard Python practice to ensure the code inside it only runs when the script is executed directly.

When you run the cell below, it will perform its first check, print the results, and then go to sleep for one hour. To stop it, you will need to interrupt the Jupyter kernel.

In [ ]:
# --- Start the Trading System ---
# This will run the main_trading_loop indefinitely.
if __name__ == "__main__":
    main_trading_loop()

INFO:root:Starting low-frequency trading system...
INFO:root:Running strategy check...
INFO:root:Data Check | Last Close: 1434.6 | 200-Day SMA: 1325.02
INFO:root:Check complete. Sleeping for 3600 seconds...


### 6. Conclusion: Choose Your Architecture

You now have two powerful and distinct design patterns in your algorithmic trading toolkit:

1.  **WebSocket Architecture**: Perfect for medium-frequency, intraday strategies that need to react to live price action in real-time.
2.  **Polling Architecture**: Ideal for low-frequency, swing, or positional strategies based on hourly, daily, or even weekly data.

Choosing the right architecture is crucial for building an efficient and effective trading system. By understanding both, you can now select the best tool for whatever strategy you want to build.